# VULCAN
## IMPORTING MODULES

In [ ]:
import pandas as pd
import numpy as np

## LOADING RAW CSVs

In [ ]:
complaints = pd.read_csv('complaints.csv')
customers = pd.read_csv('customers.csv')
service = pd.read_csv('service_records.csv')
vehicles = pd.read_csv('vehicles.csv')

print(complaints.shape, customers.shape, service.shape, vehicles.shape)

## INSPECTING THE CSVs

### complaints.csv

In [ ]:
complaints.head()

In [ ]:
complaints.describe()

In [ ]:
complaints.info()

In [ ]:
complaints.isna().sum()

In [ ]:
complaints['category'].value_counts()

No obvious category inconsistency was observed from the displayed values.

In [ ]:
complaints['status'].value_counts()

There are same categories that are being treated different just because they are written differently. We have to resolve this issue.

In [ ]:
complaints_clean = complaints.copy()

complaints_clean['status'] = complaints_clean['status'].str.strip().str.capitalize()

In [ ]:
complaints_clean['status'].value_counts()

Now, we have the 4 standard status - Resolved, Open, Escalated, Closed

***Hypothesis-1***

From this data, we can make a hypothesis that the missing values in resolution_days may be missing because those complaints have not yet been resolved — meaning complaints with an unresolved status such as Open (including open) may have resolution_days = NaN.

In [ ]:
complaints_clean[complaints_clean['resolution_days'].isna()]['status'].value_counts()

The missing values contains the values from all the categories, so we reject this hypothesis.

Now lets move on to the next investigation. We want to know if every complaint ID is unique.

In [ ]:
complaints_clean['complaint_id'].duplicated().sum()

Every complaint_id is unique. So there is no problem in this. 

Lets check if the dates are correct as they are being treated as objects right now. We want to know if they are consistantly formatted.

In [ ]:
complaints_clean['complaint_date'].head(10)

The dates appear to be in a conistant format of YYYY-MM-DD, but we need to check if it is true for the whole data. 

In [ ]:
pd.to_datetime(complaints_clean['complaint_date'], errors= 'coerce').isna().sum()

The values are consistant throughout the dataset. So we convert the dates to the proper date format.

In [ ]:
complaints_clean['complaint_date'] = pd.to_datetime(complaints_clean['complaint_date'])

In [ ]:
complaints_clean.info()

The complaint_date comlumn was succesfully converted to `datetime64` from `object`.

We have cleaned the complaint_date column and have standardized the status column. So we focus on the category column now, where 8 values are missing.

The question is - *What do these 8 missing complaints look like and can we find information in other columns regarding this.*

In [ ]:
complaints_clean[complaints_clean['category'].isna()]

From this data we were not able to find any relation between category and status. Nor were we able to guess a value from the other columns. So we will label them *Uncategorized*. 

In [ ]:
complaints_clean['category'] = complaints_clean['category'].fillna('Uncategorized')

In [ ]:
complaints_clean['category'].isna().sum()

Lets move on from the category column to the main unresolved issue - the *resolution_days* column. 

In [ ]:
complaints_clean[complaints_clean['resolution_days'].isna()]

`resolution_days` contained 63 missing values. Investigation showed that these missing values were spread across various complaint status and categories. So we cannont actually inferr the values from these fields. The missing values were therefore retained rather than artificially imputed.

In [ ]:
complaints_clean.info()

The **complaints.csv** is cleaned for now. Now we will move to the customers.csv

### customers.csv

In [ ]:
customers.head

In [ ]:
customers.info()

We can see that the `email` column has values missing. Lets see the exact number of missing values.

In [ ]:
customers.isna().sum()

There are 26 missing values in the `email` column. We will inspect the missing values and try to find a relation between other columns to see if we can reasonably fill the missing values.

In [ ]:
customers_clean = customers.copy()

In [ ]:
customers_clean[customers_clean['email'].isna()]

26 customer records have missing email addresses. After examining the affected records, the missing emails could not be reliably inferred from the available columns such as name, phone number, city, or signup date. Therefore, the missing values will not be fabricated. So we move on without changing the records. 

Now we will check if all the values in the email column are actually in the right format or not. 

In [ ]:
customers_clean.head(10)

The emails look structurally consistant. The email column is cleaned now. Lets move to the `signup_date` column. 

In [ ]:
customers_clean['signup_date'].head(10)

The `signup_date` column values are consistantly in the form of YYYY-MM-DD. Lets check if its true for the whole column.

In [ ]:
pd.to_datetime(customers_clean['signup_date'], errors= 'coerce').isna().sum()

There is no unparseable value found in the investigation. So lets change the column in the proper date format.

In [ ]:
customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'])

In [ ]:
customers_clean.info()

The data type of the *signup_date* columnn was changed from `object` to `datetime64`. 

The next column we check is the `phone` column. We need to know if all the phone numbers are right or are there anomalies in there.

In [ ]:
customers_clean['phone'].astype(str).str.len().value_counts()

After looking at the result we can say that - The 36 phone numbers that have less than 10 digits are incomplete phone numbers rather than intentionally shorter valid phone numbers. 



To test this hypothesis, we will check the rows with the incomplete phone numbers.

In [ ]:
customers_clean[customers_clean['phone'].astype(str).str.len()<10]

We cannot tell if the numbers are correct or not. The next big question is wether some of the numbers are duplicated or not?

In [ ]:
customers_clean['phone'].duplicated().sum()

Now we know that there are 36 phone numbers with less that 10 digits and 15 phone numbers that are duplicate. We will have a look at numbers that are duplicate before proceeding.

In [ ]:
customers_clean[customers_clean['phone'].duplicated(keep=False)].sort_values('phone')

Here we can see that there are many duplicate phone numbers that are caused due to duplicate records. Before deleting the duplicates, I want to see that - How many records are duplicate when we compare all the indentifying customer information?

In [ ]:
customers_clean.duplicated(subset= ['name','phone','email','city','signup_date']).sum()

In [ ]:
customers_clean[
    customers_clean['name'].str.lower().duplicated(keep=False)
].sort_values('name')

We can see that there are exact duplicates present in the data(`C9xxx` column) with different `customer_id`. 

Lets standardize the names and check for duplicates again.

In [ ]:
customers_clean['name'] = customers_clean['name'].str.strip().str.title()

In [ ]:
customers_clean.duplicated(subset=['name', 'phone', 'email', 'city', 'signup_date']).sum()

In [ ]:
customers_clean[
    customers_clean.duplicated(
        subset=['name', 'phone', 'email', 'city', 'signup_date'],
        keep=False
    )
].sort_values(['name', 'phone'])

Here we can see that `C9xxx` series has duplicate values. Before we delete this series lets confirm our hypothesis

In [ ]:
customers_clean[customers_clean['customer_id'].str.startswith('C90')].sort_values(['customer_id','name'])

In [ ]:
customers_clean[customers_clean['customer_id'].str.startswith('C90')].shape[0]

Now, it is confirmed that the `C9xxx` series contains duplicate values. So we remove the series. But before that we need to check if the series is refrenced in the vehicles.csv as there is a column `customer_id` present there.  

In [ ]:
c9_id = customers_clean[customers_clean['customer_id'].str.startswith('C9')]['customer_id']

We made `c9_id` that is basically the `C9xxx` series. Now we will check if they are refrenced in any other table before deleting the `C9xxx` series. Now we check if it is in `vehicles.csv`.

In [77]:
vehicles[
    vehicles['customer_id'].isin(c9_id)
]

,vehicle_id,customer_id,make,model,fuel_type_raw,registration_year,purchase_date,odometer_km


There is no refrence to `C9xxx` series in the `vehicles.csv`, so we are safe to delete it.

In [ ]:
customers_clean = customers_clean[~customers_clean['customer_id'].str.startswith('C90')]

We have deleted the `C9xxx` series. Lets confirm it.

In [ ]:
customers_clean.shape

In [ ]:
customers_clean.duplicated(subset=['name','phone','email','city','signup_date']).sum()

There are 400 rows present instead of 415 and no duplicates present. Hence we can confirm that the `C9xxx` series was removed. 

Now we move back to checking the phone numbers. We found that there are 36 phone numbers with less than 10 digits and there were 15 duplicates that have been taken care of. Now that the records are gone lets recheck the records.

In [ ]:
customers_clean['phone'].astype(str).str.len().value_counts().sort_index()

Here we see that there are 35 records having less than 10 digits. So we come to a hypothesis that:-

**Hypothesis** - The numbers are incomplete or invalid.

Lets check the records to confirm this hypothesis.

In [ ]:
customers_clean[customers_clean['phone'].astype(str).str.len()<10][['customer_id','name','phone','city']]

These numbers actually look like phone numbers with some digits missing. We cannot safely fill the numbers on our own, so we will accept the Hypothesis that these are invalid phone numbers. But before we make any changes, lets check if there are any duplicate phone numbers. 

In [ ]:
customers_clean.duplicated(subset='phone').sum()

There aren't any duplicate phone numbers. We change these invalid phone numbers to `unknown` to avoid confusion. Before that we will convert the data type of the column to `string` as we don't need to perform any arithematic operations on the phone number column, these are used as identifiers.

In [ ]:
customers_clean['phone'] = customers_clean['phone'].astype(str)

In [ ]:
customers_clean.loc[customers_clean['phone'].str.len()<10, 'phone'] = 'unknown'

The invalid phone numbers were changed to unkown. Now we move to the city column.

In [ ]:
customers_clean['city'].value_counts()

The city column seem to be alright. Lets check if there are any missing values before moving forward.

In [ ]:
customers_clean['city'].isna().sum()

There is no missing value present. We now move forward to the `customer_id` column.

In [ ]:
customers_clean['customer_id'].head(20)

We can see that the IDs follow a consistent `Cxxxx` pattern. Lets check if there is any ID in the column that does not follow the pattern.

In [ ]:
customers_clean[~customers_clean['customer_id'].str.match(r'^C\d{4}$')]

Now we have cleaned the customers csv file. We will run the final test before moving forward to the next CSV.

In [ ]:
customers_clean.info()

In [ ]:
customers_clean.isna().sum()

Both the test show that there are no missing values present and the values are of correct data type. Lets move on to the next dataset `service_records.csv`.

### service_records.csv

In [ ]:
service.info()

In [ ]:
service.isna().sum()

Here, we can clearly see that there are missing values present and some data columns also don't have the correct data type. 

We will continue the cleaning column wise starting from `service_id`. First we shall check if all the id's are unique.

In [ ]:
service['service_id'].duplicated().sum()

There are 10 rows with duplicate values present in the column. We need to fix this. First we will check the values that are duplicate. 

In [ ]:
service[service['service_id'].duplicated(keep=False)].sort_values('service_id')

Here we can clearly see that there are exact duplicates present in the file. We need to remove these. 

In [ ]:
service_clean = service.copy()

In [ ]:
service_clean = service_clean.drop_duplicates()

In [ ]:
service_clean.shape

The duplicates were removed. Let us verify before moving forward.

In [ ]:
service_clean.duplicated().sum()

We have confirmed that the duplicated were removed, so lets move to the next column `vehicle_id`. 

This column is a foreign key type column so duplicates will be present as a single car can have multiple service records. What we need to know is if the ID's correspond to actual vehicles. 

In [ ]:
service_clean[~service_clean['vehicle_id'].isin(vehicles['vehicle_id'])]

Every vehicle id present in service_records.csv has a corresponding vehicle in the vehicles.csv. So, we can treat `service_clean[\'vehicle_id\']` as a foreign key refferancing to `vehicles[\'vehicle_id\']`.  

Now we move to the next column - `service_date`. \
First we need to check if the dates are in correct format.

In [ ]:
service_clean['service_date'].head(20)

There is NaN present for dates that are not available. We want to check if there is are any invalid or missing dates(apart from the 40 we know are missing) present in the column.

In [ ]:
pd.to_datetime(service_clean['service_date'], errors='coerce').isna().sum()

There is no missing date present apart from the ones we know. So now we can properly convert the the column to datetime type.

In [ ]:
service_clean['service_date'] = pd.to_datetime(service_clean['service_date'])

In [ ]:
service_clean['service_date'].info

The column was successfully converted to datetime.

Now we move to `service_type`.\
We will check of there are inconsistent spellings or capitalization.

In [ ]:
service_clean['service_type'].value_counts()


No anomaly is found in this column so we move forward to the `technician` column. This column has 236 missing values. 

In [ ]:
service_clean['technician'].value_counts()